El objetivo de este experimento es implementar una versión mínima del pipeline de post-entrenamiento con GRPO para observar, aunque sea en escala reducida, la interacción entre:

- modelo base,
- prompts,
- recompensas verificables,
- y actualización de la política.



La versión que se implementará aquí será deliberadamente pequeña. Esto significa que se trabajará con:

- un modelo base reducido o accesible,
- un subconjunto muy pequeño del dataset,
- pocas iteraciones de entrenamiento,
- pocas completaciones por prompt,
- y funciones de recompensa simples.

Esta decisión no altera la lógica del método. Lo que cambia es únicamente la escala del experimento. 

In [1]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MAX_SAMPLES = 12
NUM_GENERATIONS = 2
MAX_PROMPT_LENGTH = 128
MAX_COMPLETION_LENGTH = 48
OUTPUT_DIR = "./grpo_minimo_outputs"
SEED = 42

La elección de un modelo pequeño y de un subconjunto reducido responde a una razón práctica: el propósito de este notebook es obtener una corrida viable, controlada y analizable

In [2]:
import os
import random
import numpy as np
import re
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig, TaskType

/Users/nathcab/.pyenv/versions/llama-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def fijar_semilla(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

fijar_semilla(SEED)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("semilla fija:", SEED)
print("se usa", device)

semilla fija: 42
se usa mps


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
)

model.to(device)
print("Modelo y tokenizador cargados correctamente.")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 704.03it/s]


Modelo y tokenizador cargados correctamente.


In [5]:
ejemplos = [
    {"prompt": "Resuelve y da solo la respuesta final:\n2 + 3 =", "target": "5"},
    {"prompt": "Resuelve y da solo la respuesta final:\n7 + 8 =", "target": "15"},
    {"prompt": "Resuelve y da solo la respuesta final:\n12 - 4 =", "target": "8"},
    {"prompt": "Resuelve y da solo la respuesta final:\n9 - 6 =", "target": "3"},
    {"prompt": "Resuelve y da solo la respuesta final:\n3 * 4 =", "target": "12"},
    {"prompt": "Resuelve y da solo la respuesta final:\n5 * 6 =", "target": "30"},
    {"prompt": "Resuelve y da solo la respuesta final:\n8 / 2 =", "target": "4"},
    {"prompt": "Resuelve y da solo la respuesta final:\n15 / 3 =", "target": "5"},
    {"prompt": "Resuelve y da solo la respuesta final:\n10 + 15 =", "target": "25"},
    {"prompt": "Resuelve y da solo la respuesta final:\n20 - 7 =", "target": "13"},
    {"prompt": "Resuelve y da solo la respuesta final:\n6 * 7 =", "target": "42"},
    {"prompt": "Resuelve y da solo la respuesta final:\n18 / 6 =", "target": "3"},
]

dataset = Dataset.from_list(ejemplos)
dataset = dataset.select(range(min(len(dataset), MAX_SAMPLES)))
dataset

Dataset({
    features: ['prompt', 'target'],
    num_rows: 12
})

In [6]:
peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",
)

In [7]:
def extraer_ultimo_numero(texto: str):
    coincidencias = re.findall(r"-?\d+(?:\.\d+)?", texto)
    if not coincidencias:
        return None
    return coincidencias[-1]

def reward_exactitud(completions, target, **kwargs):
    rewards = []
    for completion, esperado in zip(completions, target):
        if isinstance(completion, list):
            texto = completion[0]["content"]
        else:
            texto = str(completion)

        pred = extraer_ultimo_numero(texto)
        rewards.append(1.0 if pred == esperado else 0.0)
    return rewards

def reward_formato(completions, **kwargs):
    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            texto = completion[0]["content"].strip()
        else:
            texto = str(completion).strip()
        rewards.append(0.2 if len(texto.split()) <= 8 else 0.0)
    return rewards

In [8]:
training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    num_generations=2,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=5e-6,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    report_to="none",
    remove_unused_columns=False,
    use_cpu=(device == "cpu"),
    fp16=False,
    bf16=False,
    gradient_checkpointing=False,
)

In [9]:
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_exactitud, reward_formato],
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
)
print("GRPOTrainer inicializado correctamente.")

GRPOTrainer inicializado correctamente.


In [10]:
resultado_entrenamiento = trainer.train()
resultado_entrenamiento

/Users/nathcab/.pyenv/versions/llama-env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.353512
6,0.000000
7,0.000000
8,0.000000
9,0.000000
10,0.000000


TrainOutput(global_step=12, training_loss=0.029459312558174133, metrics={'train_runtime': 64.2515, 'train_samples_per_second': 0.187, 'train_steps_per_second': 0.187, 'total_flos': 0.0, 'train_loss': 0.029459312558174133})

In [11]:
print("Resumen del entrenamiento:")
print("Global step:", resultado_entrenamiento.global_step)
print("Training loss:", resultado_entrenamiento.training_loss)
print("Metrics:", resultado_entrenamiento.metrics)

Resumen del entrenamiento:
Global step: 12
Training loss: 0.029459312558174133
Metrics: {'train_runtime': 64.2515, 'train_samples_per_second': 0.187, 'train_steps_per_second': 0.187, 'total_flos': 0.0, 'train_loss': 0.029459312558174133}


Además de registrar las métricas numéricas, conviene observar directamente algunas respuestas generadas por el modelo después del ajuste. Aunque esta corrida es mínima y no pretende producir una mejora espectacular, sí permite inspeccionar el tipo de salidas que el modelo produce tras completar el ciclo de entrenamiento.

In [18]:
def generar_respuesta(prompt, max_new_tokens=32):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    texto = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return texto

prompts_prueba = [
    "Resuelve y da solo la respuesta final:\n2 + 3 =",
    "Resuelve y da solo la respuesta final:\n6 * 7 =",
    "Resuelve y da solo la respuesta final:\n20 - 7 =",
    "Resuelve y da solo la respuesta final:\n15 / 3 =",
]

for p in prompts_prueba:
    print("PROMPT:")
    print(generar_respuesta(p))


PROMPT:
Resuelve y da solo la respuesta final:
2 + 3 = 5

La respuesta final es: 5

La respuesta final es: 5

La respuesta final es
PROMPT:
Resuelve y da solo la respuesta final:
6 * 7 = 42

La respuesta final es: 42

La respuesta final es: 42

La respu
PROMPT:
Resuelve y da solo la respuesta final:
20 - 7 = 17

La respuesta final es: 17

La respuesta final es: 17

La respu
PROMPT:
Resuelve y da solo la respuesta final:
15 / 3 = 6.5

The answer is: 6.5


1. El pipeline experimental logró ejecutarse de principio a fin en una configuración reducida.

2. Fue necesario reducir el tamaño del modelo, el número de generaciones por prompt y la longitud máxima de completación para adaptarlo a las restricciones de memoria del entorno local.

3. La corrida obtenida debe interpretarse como una demostración funcional del método y no como una evaluación exhaustiva del potencial del algoritmo.

4. Las recompensas utilizadas en esta prueba fueron deliberadamente simples, por lo que la señal de aprendizaje resultante también es limitada.

- se trabajó con un modelo pequeño;

- se utilizó un dataset artificial y muy reducido;

- el número de épocas y generaciones por prompt fue mínimo;

- las funciones de recompensa fueron simples y diseñadas solo para verificar el funcionamiento del pipeline;

- el entrenamiento se ejecutó bajo restricciones de memoria propias del entorno local.

Por estas razones, los resultados no deben interpretarse como una medición definitiva del desempeño de GRPO, sino como evidencia de que la tubería básica de post-entrenamiento puede implementarse y ejecutarse en una versión mínima reproducible.